# Ôn tập Coding 10 điểm – Online Retail

**MSSV:** 24729251  
**Họ tên:** Lý Ngọc Anh Thư  
**Số máy:** 03

> Bản này viết lại theo cách code cơ bản đã học: dùng `df`, `df2`, `groupby()`, `agg()`, `pivot_table()`, `.loc`, `isna()`, `fillna()`...  
> Các dòng `# note:` giải thích ngay bên dưới phần code để dễ ôn và nhớ.

## 1. Import thư viện và đọc dữ liệu

Dữ liệu `OnlineRetail.csv` phải nằm cùng thư mục với notebook. File dùng encoding `latin1` để đọc đúng ký tự.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Đọc file CSV
df = pd.read_csv('OnlineRetail.csv', encoding='latin1')

# In 5 dòng cuối
df.tail()


- `pd.read_csv()` dùng để đọc file CSV và tạo DataFrame.
- `encoding='latin1'` giúp đọc file này mà không bị lỗi mã hóa ký tự.
- `df.tail()` mặc định hiển thị 5 dòng cuối.


## 2. Discovery & Structuring

### 2.1. Xem thông tin dữ liệu

In [ ]:
# Xem kiểu dữ liệu và số giá trị không thiếu của từng cột
df.info()#* đọc thông tin data (0.15đ)

# Số dòng
print(df.shape[0])

# Số cột
print(df.shape[1])

# * in ra số quốc gia (Country) trong bộ data và 4 quốc gia có nhiều dòng nhất (0.15đ):
# Số quốc gia khác nhau
print(df['Country'].nunique())

# 4 quốc gia có nhiều dòng nhất
df['Country'].value_counts().head(4)


**Note:**
- `df.info()` xem tên cột, kiểu dữ liệu, số giá trị không thiếu.
- `df.shape[0]` là số dòng; `df.shape[1]` là số cột.
- `nunique()` đếm số giá trị khác nhau.
- `value_counts()` đếm số lần xuất hiện của từng giá trị; `head(4)` lấy 4 giá trị đầu.


### 2.2. Đổi tên cột

In [ ]:
# * đổi tên 4 cột: InvoiceDate --> invoice_time, UnitPrice --> unit_price, CustomerID --> cust_id, Country --> country (0.2đ)
df = df.rename(columns={
    'InvoiceDate': 'invoice_time',
    'UnitPrice': 'unit_price',
    'CustomerID': 'cust_id',
    'Country': 'country'
})

# In danh sách cột sau khi đổi tên
print(df.columns.tolist()) #* in lại danh sách các cột sau khi rename (0.15đ)


**Note:** `rename(columns={...})` đổi tên cột. Những cột không ghi trong phần `rename` vẫn giữ nguyên tên cũ.

### 2.3. Tạo bảng profile_df

Bảng gồm kiểu dữ liệu, số giá trị khác nhau, số lượng thiếu và tỷ lệ thiếu. Sắp xếp giảm dần theo số lượng thiếu.


In [ ]:
# * Tạo dataframe profile_df bao gồm dtype, nunique, missing_count, missing_percentage. Sắp xếp giảm dần theo missing_count. (0.5đ)
profile_df = pd.DataFrame({
    'dtype': df.dtypes,
    'nunique': df.nunique(),
    'missing_count': df.isna().sum(),
    'missing_percentage': df.isna().mean() * 100
})

profile_df = profile_df.sort_values(
    by='missing_count',
    ascending=False
)

profile_df


**Note:**
- `df.dtypes`: kiểu dữ liệu của từng cột.
- `df.nunique()`: số giá trị khác nhau của từng cột.
- `df.isna().sum()`: đếm số ô thiếu theo từng cột.
- `df.isna().mean() * 100`: tính tỷ lệ thiếu (%). Vì True được tính là 1 và False là 0 nên lấy mean sẽ ra tỷ lệ.
- `ascending=False`: sắp xếp giảm dần.


### 2.4. Lọc theo điều kiện

In [ ]:
# * tìm số dòng thỏa điều kiện country là France và Quantity từ 10 trở lên (0.2đ)
# Số dòng có country là France và Quantity từ 10 trở lên
condition = (df['country'] == 'France') & (df['Quantity'] >= 10)
print(condition.sum())

# * cho biết số dòng có InvoiceNo bắt đầu bằng chữ C (hóa đơn hủy) và country khác United Kingdom (0.2đ). Gợi ý: .str.startswith()
# Số dòng có InvoiceNo bắt đầu bằng C và country khác United Kingdom
condition = (df['InvoiceNo'].astype(str).str.startswith('C')) & (df['country'] != 'United Kingdom')
print(condition.sum())


**Note:**
- `&` nghĩa là đồng thời thỏa mãn cả hai điều kiện.
- Mỗi điều kiện phải đặt trong ngoặc.
- `condition.sum()` đếm số dòng có điều kiện True.
- `str.startswith('C')` kiểm tra chuỗi có bắt đầu bằng chữ C hay không.
- Lưu ý: phải kiểm tra `InvoiceNo`, không phải `invoice_time`, vì mã hóa đơn mới là cột bắt đầu bằng C.


## 3. Cleaning – Duplicated & Missing Data

### 3.1. Kiểm tra và xóa dòng trùng

Duplicate count (0.4đ)
* In số dòng trùng cả bản ghi.
* In thêm số dòng trùng theo cặp cột InvoiceNo và Description (một sản phẩm xuất hiện nhiều lần trong cùng hóa đơn). Gợi ý: subset=[...]

In [ ]:
# Số dòng trùng toàn bộ bản ghi
print(df.duplicated().sum())

# Số dòng trùng theo cặp InvoiceNo và Description
print(df.duplicated(subset=['InvoiceNo', 'Description']).sum())

# Xóa các dòng trùng toàn bộ bản ghi
df = df.drop_duplicates()

# Kiểm tra lại
print(df.duplicated().sum())
print(df.shape)


**Note:**
- `duplicated()` kiểm tra dòng trùng.
- `subset=[...]` chỉ xét các cột được chỉ định để xác định trùng.
- `drop_duplicates()` xóa các dòng trùng toàn bộ bản ghi.
- `df.shape` kiểm tra lại số dòng và số cột sau khi xóa.
- Đề chỉ yêu cầu xóa trùng toàn bộ bản ghi, không yêu cầu xóa trùng theo InvoiceNo và Description.


### 3.2. Tạo missing_report

missing_report (0.4đ)
* Kiểm tra dữ liệu có missing không?
* Nếu có, tạo missing_report gồm 3 cột: column_name, missing_count, missing_percentage
* missing_percentage làm tròn 2 chữ số thập phân
* Sắp xếp missing_report tăng dần theo missing_percentage
* Chỉ giữ các dòng có missing_count khác 0

In [ ]:
# Đếm số giá trị thiếu ở từng cột
missing_count = df.isna().sum()

# Tạo bảng missing_report
missing_report = pd.DataFrame({
    'column_name': df.columns,
    'missing_count': missing_count.values,
    'missing_percentage': (df.isna().mean().values * 100).round(2)
})

# Chỉ giữ những cột có giá trị thiếu
missing_report = missing_report[missing_report['missing_count'] != 0]

# Sắp xếp tăng dần theo tỷ lệ thiếu
missing_report = missing_report.sort_values(
    by='missing_percentage',
    ascending=True
)

missing_report


**Note:**
- `df.isna().sum()` đếm số ô thiếu ở từng cột.
- `df.isna().mean() * 100` tính phần trăm thiếu.
- `round(2)` làm tròn 2 chữ số thập phân.
- Điều kiện `missing_count != 0` loại các cột không có dữ liệu thiếu.
- `ascending=True` là sắp xếp tăng dần.


### 3.3. Vẽ biểu đồ tỷ lệ thiếu

Missing-value bar chart (0.4đ)
* Vẽ bar chart cho missing_report với x là column_name, y là missing_percentage
* xlabel là tên cột, ylabel là tỷ lệ thiếu (%)

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    missing_report['column_name'],
    missing_report['missing_percentage']
)

plt.xlabel('Tên cột')
plt.ylabel('Tỷ lệ thiếu (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


**Note:**
- `plt.bar(x, y)` vẽ biểu đồ cột.
- Trục X là tên cột; trục Y là tỷ lệ thiếu.
- `rotation=45` xoay tên cột để dễ đọc.
- `tight_layout()` giúp nhãn không bị cắt.


### 3.4. Tạo cust_missing và tính tỷ lệ thiếu theo 4 quốc gia

In [ ]:
# True nếu cust_id bị thiếu, False nếu không thiếu
df['cust_missing'] = df['cust_id'].isna() #* tạo cột cust_missing cho biết cust_id có bị thiếu hay không (True/False hoặc 1/0) (0.25đ)


# * cust_missing_rate = Chỉ xét 4 quốc gia có nhiều dòng nhất. 
# Nhóm theo country, tính mean của cột cust_missing rồi nhân 100 
# để ra tỷ lệ thiếu (%). Sắp xếp giảm dần và đổi tên thành cust_missing_pct. (0.25đ):

# Lấy tên 4 quốc gia có nhiều dòng nhất
top4_countries = df['country'].value_counts().head(4).index

# Chỉ giữ 4 quốc gia đó
df_top4 = df[df['country'].isin(top4_countries)]

# Nhóm theo country và tính tỷ lệ thiếu (%)
cust_missing_rate = df_top4.groupby('country')['cust_missing'].mean() * 100

# Sắp xếp giảm dần
cust_missing_rate = cust_missing_rate.sort_values(ascending=False)

# Đổi tên Series
cust_missing_rate = cust_missing_rate.rename('cust_missing_pct')

cust_missing_rate


**Note:**
- `isna()` tạo kết quả True/False cho biết ô có bị thiếu không.
- `value_counts().head(4).index` lấy tên 4 quốc gia xuất hiện nhiều nhất.
- `isin()` kiểm tra giá trị có nằm trong danh sách hay không.
- `groupby('country')` gom các dòng theo quốc gia.
- Lấy `mean()` của cột True/False rồi nhân 100 sẽ ra tỷ lệ thiếu (%).
- `rename()` đổi tên Series.


### 3.5. Vẽ biểu đồ ngang tỷ lệ thiếu cust_id

* Vẽ bar chart ngang tỷ lệ thiếu cust_id theo country. xlabel là tỷ lệ thiếu (%), ylabel là quốc gia. (0.25đ)

In [ ]:
plt.figure(figsize=(8, 4))

plt.barh(
    cust_missing_rate.index,
    cust_missing_rate.values
)

plt.xlabel('Tỷ lệ thiếu (%)')
plt.ylabel('Quốc gia')
plt.tight_layout()
plt.show()


**Note:** `plt.barh()` vẽ biểu đồ cột ngang. `index` là tên quốc gia, `values` là tỷ lệ thiếu tương ứng.

### 3.6. Điền giá trị thiếu cho Description


Fill missing (0.4đ)
* Description: điền 'NO_NAME'.
* Kiểm tra lại

In [ ]:
df['Description'] = df['Description'].fillna('NO_NAME')

# Kiểm tra lại số giá trị thiếu
print(df['Description'].isna().sum())


**Note:**
- `fillna('NO_NAME')` thay các ô thiếu trong Description bằng chữ `NO_NAME`.
- Sau khi điền, dùng `isna().sum()` để kiểm tra còn thiếu bao nhiêu ô.


### 3.7. Tạo is_return và lọc giao dịch hợp lệ

Invalid values (0.2đ)
* Quantity <= 0 (hàng trả lại / hóa đơn hủy) và unit_price <= 0 (hàng tặng / điều chỉnh sổ sách) không phải giao dịch bán hàng thật sự.
* Tạo cột is_return: bằng 1 nếu InvoiceNo bắt đầu bằng chữ C, ngược lại bằng 0. (gợi ý startswith)
* Sau đó giữ lại trong df các dòng có Quantity > 0 và unit_price > 0. In shape trước và sau khi lọc.

In [ ]:
# Tạo is_return: 1 nếu InvoiceNo bắt đầu bằng C, ngược lại là 0
df['is_return'] = df['InvoiceNo'].astype(str).str.startswith('C').astype(int)

# Xem số dòng trước khi lọc
print('Shape trước khi lọc:', df.shape)

# Chỉ giữ Quantity > 0 và unit_price > 0
df = df[(df['Quantity'] > 0) & (df['unit_price'] > 0)].copy()

# Xem số dòng sau khi lọc
print('Shape sau khi lọc:', df.shape)


**Note:**
- `startswith('C')` trả về True/False.
- `astype(int)` đổi True thành 1 và False thành 0.
- Điều kiện `Quantity > 0` loại số lượng bằng 0 hoặc âm.
- Điều kiện `unit_price > 0` loại đơn giá bằng 0 hoặc âm.
- `.copy()` tạo DataFrame riêng sau khi lọc.
- Cột `is_return` được tạo trước khi lọc để giữ lại thông tin mã hóa đơn bắt đầu bằng C.


# !!!Tới đây thoi, còn bên dưới chưa sửa!!!

## 4. Cleaning – Outlier

### 4.1. Tạo num_df

In [ ]:
# Chỉ lấy hai cột Quantity và unit_price
num_df = df[['Quantity', 'unit_price']].copy()

num_df.head()


**Note:** Chọn hai cột cần kiểm tra outlier và dùng `.copy()` để tạo DataFrame riêng.

### 4.2. Tính Q1, Q3, IQR và giới hạn outlier

Đề yêu cầu Q1 là phân vị 15%, Q3 là phân vị 85%.


In [ ]:
# Tính Q1 và Q3 cho từng cột
Q1 = num_df.quantile(0.15)
Q3 = num_df.quantile(0.85)

# Tính IQR
IQR = Q3 - Q1

# Tính giới hạn dưới và giới hạn trên
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Đếm số outlier của từng cột
outlier_count = ((num_df < lower_bound) | (num_df > upper_bound)).sum()

# Ghép kết quả thành bảng
bound_table = pd.DataFrame({
    'Q1': Q1,
    'Q3': Q3,
    'IQR': IQR,
    'lower_bound': lower_bound,
    'upper_bound': upper_bound,
    'outlier_count': outlier_count
})

bound_table


**Note:**
- `quantile(0.15)` lấy phân vị 15%; `quantile(0.85)` lấy phân vị 85%.
- `IQR = Q3 - Q1`.
- Giới hạn dưới = Q1 − 1.5 × IQR.
- Giới hạn trên = Q3 + 1.5 × IQR.
- Giá trị nhỏ hơn giới hạn dưới hoặc lớn hơn giới hạn trên được xem là outlier.
- `|` nghĩa là HOẶC.
- Các biến Q1, Q3, IQR, lower_bound, upper_bound là Series, mỗi cột có một giá trị giới hạn riêng.


### 4.3. Lọc bỏ dòng có ít nhất một cột là outlier

In [ ]:
# Kiểm tra Quantity có nằm trong giới hạn không
condition1 = df['Quantity'].between(
    lower_bound['Quantity'],
    upper_bound['Quantity']
)

# Kiểm tra unit_price có nằm trong giới hạn không
condition2 = df['unit_price'].between(
    lower_bound['unit_price'],
    upper_bound['unit_price']
)

# Chỉ giữ dòng mà cả hai cột đều nằm trong giới hạn
df_filtered = df[condition1 & condition2].copy()

print('Shape trước lọc outlier:', df.shape)
print('Shape sau lọc outlier:', df_filtered.shape)


**Note:**
- `between(a, b)` kiểm tra giá trị có nằm trong đoạn từ a đến b hay không (bao gồm hai đầu).
- `condition1 & condition2` yêu cầu cả Quantity và unit_price đều nằm trong giới hạn.
- Dòng nào có ít nhất một cột nằm ngoài giới hạn sẽ bị loại.
- Dùng `lower_bound['Quantity']` để lấy giới hạn riêng của cột Quantity; tương tự với unit_price.


### 4.4. Dùng clip để chặn outlier, không xóa dòng

In [ ]:
df_clipped = df.copy()

df_clipped['Quantity'] = df_clipped['Quantity'].clip(
    lower=lower_bound['Quantity'],
    upper=upper_bound['Quantity']
)

df_clipped['unit_price'] = df_clipped['unit_price'].clip(
    lower=lower_bound['unit_price'],
    upper=upper_bound['unit_price']
)

df_clipped[['Quantity', 'unit_price']].head()


**Note:**
- `clip()` không xóa dòng.
- Giá trị nhỏ hơn giới hạn dưới được kéo lên bằng giới hạn dưới.
- Giá trị lớn hơn giới hạn trên được kéo xuống bằng giới hạn trên.
- Phải dùng ngưỡng riêng cho từng cột.


### 4.5. Histogram của unit_price trong df_filtered

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(df_filtered['unit_price'].dropna(), bins=50)

plt.title('phân phối unit_price')
plt.xlabel('Đơn giá (GBP)')
plt.ylabel('Tần số')
plt.tight_layout()
plt.show()


**Note:**
- `plt.hist()` vẽ histogram để xem phân phối dữ liệu.
- `dropna()` bỏ giá trị thiếu trước khi vẽ.
- `bins=50` chia dữ liệu thành 50 khoảng.


### 4.6. Skew, kurtosis, mean và median

In [ ]:
price = df_filtered['unit_price'].dropna()

print('Skew:', price.skew())
print('Kurtosis:', price.kurt())
print('Mean:', price.mean())
print('Median:', price.median())


**Cách nhận xét:**
- Skew > 0: phân phối lệch phải.
- Skew < 0: phân phối lệch trái.
- Skew gần 0: phân phối tương đối đối xứng.
- Mean lớn hơn Median thường là dấu hiệu lệch phải; Mean nhỏ hơn Median thường là dấu hiệu lệch trái.
- `kurt()` của Pandas là excess kurtosis:
  - Kurtosis > 0: đỉnh nhọn hơn và đuôi dày hơn phân phối chuẩn.
  - Kurtosis < 0: đỉnh bẹt hơn và đuôi mỏng hơn phân phối chuẩn.
  - Kurtosis = 0: tương đương phân phối chuẩn về excess kurtosis.
- Không tự ghi số nhận xét mẫu; phải dựa vào kết quả chạy trên dữ liệu của mình.


### 4.7. Tạo price_log bằng log1p

In [ ]:
df_filtered['price_log'] = np.log1p(df_filtered['unit_price'])

df_filtered[['unit_price', 'price_log']].head()


**Note:**
- `np.log1p(x)` tính ln(1 + x).
- Ở bước trước đã lọc `unit_price > 0`, nên dữ liệu phù hợp để tính log1p.
- Khác với `np.log(x)`, `log1p` cộng 1 trước khi tính log.


## 5. Univariate Analysis

### 5.1. Thống kê mô tả

In [ ]:
df_filtered[['Quantity', 'unit_price', 'price_log']].describe()


**Note:** `describe()` cho biết count, mean, std, min, các tứ phân vị và max của các cột số.

### 5.2. Thống kê price_log và vẽ hai histogram


In [ ]:
price_log = df_filtered['price_log'].dropna()

print('Skew:', price_log.skew())
print('Kurtosis:', price_log.kurt())
print('Mean:', price_log.mean())
print('Median:', price_log.median())

plt.figure(figsize=(10, 4))

# Histogram unit_price
plt.subplot(1, 2, 1)
plt.hist(df_filtered['unit_price'].dropna(), bins=50)
plt.title('unit_price')

# Histogram price_log
plt.subplot(1, 2, 2)
plt.hist(df_filtered['price_log'].dropna(), bins=50)
plt.title('price_log')

plt.tight_layout()
plt.show()


**Note:**
- `subplot(1, 2, 1)`: chia khu vực vẽ thành 1 hàng, 2 cột và chọn vị trí thứ nhất.
- `subplot(1, 2, 2)`: chọn vị trí thứ hai.
- So sánh hai biểu đồ để xem phép log1p làm thay đổi hình dạng phân phối như thế nào.


## 6. Bivariate & Multivariate Analysis

### 6.1. Feature engineering – tạo biến mới

In [ ]:
# Đổi invoice_time sang kiểu ngày giờ
df_filtered['invoice_time'] = pd.to_datetime(df_filtered['invoice_time'])

# Tính doanh thu từng dòng
df_filtered['sales_value'] = df_filtered['Quantity'] * df_filtered['unit_price']

# Lấy tháng
df_filtered['month_no'] = df_filtered['invoice_time'].dt.month

# Lấy giờ
df_filtered['hour_of_day'] = df_filtered['invoice_time'].dt.hour

# Lấy thứ trong tuần
df_filtered['day_name'] = df_filtered['invoice_time'].dt.day_name()

df_filtered[['invoice_time', 'sales_value', 'month_no', 'hour_of_day', 'day_name']].head()


**Note:**
- `pd.to_datetime()` chuyển dữ liệu sang kiểu ngày giờ.
- `.dt.month` lấy tháng.
- `.dt.hour` lấy giờ.
- `.dt.day_name()` lấy tên thứ trong tuần.
- `sales_value = Quantity * unit_price` tính giá trị bán của từng dòng hàng.


### 6.2. Correlation heatmap

In [ ]:
corr_cols = ['Quantity', 'unit_price', 'sales_value', 'month_no', 'hour_of_day']

corr = df_filtered[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')

plt.title('Correlation heatmap')
plt.tight_layout()
plt.show()


**Note:**
- `.corr()` tính hệ số tương quan giữa các cột số.
- `annot=True` hiển thị số trong từng ô.
- `fmt='.2f'` hiển thị 2 chữ số thập phân.
- Tương quan không chứng minh quan hệ nhân quả.


### 6.3. Groupby theo day_name

In [ ]:
day_sales = df_filtered.groupby('day_name')['sales_value'].agg([
    'count',
    'mean',
    'max'
])

day_sales


**Note:**
- `groupby('day_name')` gom dữ liệu theo thứ trong tuần.
- Chọn cột `sales_value` để thống kê.
- `agg([...])` cho phép tính nhiều thống kê cùng lúc:
  - `count`: số giá trị không thiếu.
  - `mean`: trung bình.
  - `max`: lớn nhất.


### 6.4. Groupby theo 4 quốc gia có nhiều dòng nhất

In [ ]:
country_sales = df_filtered[df_filtered['country'].isin(top4_countries)]

country_sales = country_sales.groupby('country')['sales_value'].agg([
    'count',
    'mean'
])

country_sales = country_sales.sort_values(
    by='mean',
    ascending=False
)

country_sales


**Note:**
- Dùng lại `top4_countries` đã tạo ở phần missing data.
- `isin(top4_countries)` chỉ giữ 4 quốc gia đó.
- `groupby('country')` gom theo quốc gia.
- `agg(['count', 'mean'])` tính số lượng và giá trị trung bình.
- Sắp xếp giảm dần theo mean.


### 6.5. Pivot table và heatmap

In [ ]:
# Chỉ giữ Germany, France và Spain
df_three_countries = df_filtered[
    df_filtered['country'].isin(['Germany', 'France', 'Spain'])
]

# Tạo pivot table: hàng là tháng, cột là quốc gia
pivot_sales = df_three_countries.pivot_table(
    index='month_no',
    columns='country',
    values='sales_value',
    aggfunc='median'
)

pivot_sales


In [ ]:
plt.figure(figsize=(8, 6))

sns.heatmap(
    pivot_sales,
    annot=True,
    fmt='.2f',
    cmap='YlGnBu'
)

plt.title('Median sales_value theo tháng và quốc gia')
plt.xlabel('Quốc gia')
plt.ylabel('Tháng')
plt.tight_layout()
plt.show()


**Note:**
- `pivot_table()` tạo bảng tổng hợp theo hàng và cột.
- `index='month_no'`: tháng nằm ở hàng.
- `columns='country'`: quốc gia nằm ở cột.
- `values='sales_value'`: dữ liệu cần tính.
- `aggfunc='median'`: lấy trung vị nếu một nhóm có nhiều giá trị.
- Heatmap giúp so sánh median sales_value giữa các tháng và quốc gia.


## 7. Publishing – Xuất file

### 7.1. Lưu 500 dòng đầu ra Excel

In [ ]:
df_filtered.head(500).to_excel('shop_final.xlsx', index=False)


**Note:**
- `head(500)` lấy 500 dòng đầu.
- `to_excel()` lưu thành file Excel.
- `index=False` không lưu cột số thứ tự index.


### 7.2. Lưu 500 dòng cuối ra HTML

In [ ]:
df_filtered.tail(500).to_html('shop_final.html', index=False)


**Note:**
- `tail(500)` lấy 500 dòng cuối.
- `to_html()` lưu DataFrame thành bảng HTML.
- `index=False` không xuất cột index.

---

## Những chỗ cần nhớ khi làm bài

1. Đọc file đúng encoding: `encoding='latin1'`.
2. Mã hóa đơn nằm ở `InvoiceNo`; kiểm tra hóa đơn hủy bằng `str.startswith('C')`.
3. Tỷ lệ missing: `df.isna().mean() * 100`.
4. Đếm số dòng thỏa điều kiện: tạo `condition`, sau đó dùng `condition.sum()`.
5. Lọc nhiều điều kiện cùng lúc: dùng `&`, mỗi điều kiện đặt trong ngoặc.
6. `drop_duplicates()` xóa dòng trùng; `drop_duplicates(subset=[...])` chỉ xét các cột được chỉ định.
7. `fillna()` dùng để điền giá trị thiếu.
8. `between()` kiểm tra một giá trị có nằm trong khoảng hay không.
9. `clip()` chặn giá trị ở giới hạn dưới và giới hạn trên, không xóa dòng.
10. `groupby()` dùng để nhóm; `agg()` dùng để tính nhiều thống kê.
11. `pivot_table()` tạo bảng tổng hợp theo hàng và cột.
12. `to_excel()` và `to_html()` dùng để xuất file.
